# PiCo — TransNEO clinical transfer demo

Reproduces **Fig. 5c** (residual cancer burden, RCB — regression) and
**Fig. 5d** (pCR — classification) from Kirkham et al. (2025), plus the
corresponding permutation-feature-importance panels for the best Clinical+z+RNA
model.

The notebook is focused on modelling: for every (target × feature-set × encoder
× seed) it forward-passes the bundled iCoVAE / VAE encoder on the LINCS1000
expression matrix of each cohort, refits an ElasticNet (RCB) or
LogisticRegression (pCR) with the saved hyperparameters, and scores on the
external-validation ARTemis+PBCP cohort. Plotting code is imported from
`src/utils/plot_helpers.py` so the plots match the paper exactly.

Six feature-set variants are shown — Clinical, RNA, Clinical+RNA, `z`,
Clinical+`z`, Clinical+`z`+RNA — across two encoders, two targets, and 10
seeds (240 probe fits total). Expected runtime: **~3 min** (+ ~30 s for the repo clone).

In [ ]:
# --- Setup: Colab pip install + clone; local no-op ---
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Clone the repo (with bundled demo assets) and pin a torch version
    # compatible with the current Colab runtime.
    !git clone --depth 1 https://github.com/domkirkham/pico.git
    !pip -q install torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu
    %cd pico

In [ ]:
# --- Imports + path setup ---
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# ElasticNet/LogReg with l1_ratio=0 print a "use Ridge instead" warning per fit;
# safe to silence — the bundled hyperparameters are the paper's, not ours.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

REPO_ROOT = Path().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import models.pico  # noqa: F401 — registers TabularEncoder for torch.load
from utils.plot_helpers import (
    apply_paper_style, live_permutation_fi, plot_transneo_perf,
)
from utils.comp_utils import plot_feat_imps

apply_paper_style(REPO_ROOT / "src" / "fonts")

DEMO_ROOT = REPO_ROOT / "demo" / "assets"
OUT_DIR = REPO_ROOT / "demo" / "outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT = "artemis_pbcp"
SEEDS = list(range(10, 110, 10))

# Six feature-set variants from the paper. Each maps to its Stage-2 dir suffix.
FEATURE_SETS = [
    ("Clinical",         "_Size.at.diagnosis_7_norep",  True),
    ("RNA",              "_PGR.log2.tpm_11_norep",      True),
    ("Clinical+RNA",     "_Size.at.diagnosis_18_norep", True),
    ("Rep",              "",                             False),
    ("Clinical+Rep",     "_Size.at.diagnosis_7",        False),
    ("Clinical+Rep+RNA", "_Size.at.diagnosis_18",       False),
]
ENCODERS = [("icovae_MCL1_16", "PiCo"), ("vae", "VAE")]

## 1. Inspect the bundled inputs

The TransNEO demo reads four small files:

* **`transneo_expression_lincs918.csv.gz`** — TransNEO (training) cohort
  expression on the LINCS1000 gene set.
* **`artemis_pbcp_expression_lincs918.csv.gz`** — ARTemis+PBCP
  (external-validation) cohort, same gene set.
* **`transneo_clinical_plus_rna.csv` / `artemis_pbcp_clinical_plus_rna.csv`**
  — clinical covariates (age, tumour size, receptor status, …) + precomputed
  RNA features (PGR expression, immune signatures, …). Columns match the
  `confounders` field of each Stage-2 `args_best_s{seed}.txt`.

In [ ]:
# Per-seed expression matrices: each iCoVAE encoder was trained with the
# variance filter applied to its specific KFold split, so the gene column
# order differs per seed. The bundle ships one matrix per seed for each
# cohort to keep the demo bit-equivalent to the training pipeline.
def load_exp(prefix, seed):
    return pd.read_csv(
        DEMO_ROOT / "data" / f"{prefix}_expression_lincs918_s{seed}.csv.gz",
        index_col=0,
    )

clin_tn = pd.read_csv(DEMO_ROOT / "data" / "transneo_clinical_plus_rna.csv",
                      index_col=0)
clin_ap = pd.read_csv(DEMO_ROOT / "data" / "artemis_pbcp_clinical_plus_rna.csv",
                      index_col=0)
exp_tn_sample = load_exp("transneo", SEEDS[0])
exp_ap_sample = load_exp("artemis_pbcp", SEEDS[0])
print(f"TransNEO expression (seed {SEEDS[0]}):", exp_tn_sample.shape,
      "| clinical:", clin_tn.shape)
print(f"ARTemis+PBCP expression (seed {SEEDS[0]}):", exp_ap_sample.shape,
      "| clinical:", clin_ap.shape)
print("\nClinical head (TransNEO):")
print(clin_tn[["RCB.score", "resp.pCR", "Age.at.diagnosis",
               "ER.status", "HER2.status", "Size.at.diagnosis"]].head(3))

## 2. Load one encoder

The TransNEO encoders use a PiCo iCoVAE with `MCL1`-anchored constraints and
a 16-dim latent space (both values picked during Stage-1 hyperparameter
optimisation; see the stage-1 `args_best.txt`).

In [ ]:
stage1_dir = (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo"
              / "RCB.score" / EXPERIMENT / "icovae_MCL1_16")
stage1_cfg = json.loads((stage1_dir / "args_best.txt").read_text())
print(f"Stage-1 z_dim = {stage1_cfg['z_dim']}; "
      f"n_constraints = {len(stage1_cfg['curr_constraints'])}")
encoder = torch.load(stage1_dir / f"encoder_{SEEDS[0]}.pt",
                     map_location="cpu", weights_only=False)
encoder.eval()
print("\nEncoder module:")
print(encoder)

## 3. Live encoding + Stage-2 probe fit across all variants

For every (target × feature-set × encoder × seed) we build a feature matrix
that is one of:

| Variant | Feature matrix |
| --- | --- |
| Clinical | 7 clinical covariates |
| RNA | 11 RNA/signature covariates |
| Clinical+RNA | 7 + 11 = 18 covariates |
| Rep | 16-dim `z` only |
| Clinical+Rep | 7 covariates + `z` |
| Clinical+Rep+RNA | 18 covariates + `z` |

fit an ElasticNet (RCB.score, regression) or LogisticRegression (resp.pCR,
classification) with the saved `alpha`/`l1_ratio`, and compute cross-validation
metrics on TransNEO plus external metrics on ARTemis+PBCP. The Stage-2
pipeline mirrors `src/models/pico.py:PiCoSK.fit_regressor` exactly, including
the `StandardScaler` fit on the training feature matrix.

In [ ]:
def _feature_matrix(feat_set, clin_df, z_arr, confounders, samples):
    # Build the feature matrix for one variant. z_arr is row-aligned with `samples`.
    if feat_set == "Rep":
        return z_arr
    cov = clin_df.loc[samples, confounders].to_numpy(dtype=float)
    if feat_set in ("Clinical", "RNA", "Clinical+RNA"):
        return cov
    return np.concatenate([z_arr, cov], axis=1)


def _encode(enc_dir, samples, exp, seed):
    enc = torch.load(enc_dir / f"encoder_{seed}.pt",
                     map_location="cpu", weights_only=False)
    enc.eval()
    x = exp.loc[samples].to_numpy(dtype=np.float32)
    with torch.no_grad():
        loc, _ = enc(torch.as_tensor(x))
    return loc.numpy()


def _fit_and_score(target, feat_set, suffix, norep_suffix, enc_key, seed):
    reg_name = "ElasticNet" if target == "RCB.score" else "LogisticRegression"
    stage2_dir = (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo"
                  / target / EXPERIMENT / "pico" / f"{reg_name}_{enc_key}{suffix}")
    cfg = json.loads((stage2_dir / f"args_best_s{seed}.txt").read_text())

    # Per-seed expression in the variance-sorted column order the encoder
    # was trained on (see preprocess_transneo_data in build_demo_bundle.py).
    exp_tn = load_exp("transneo", seed)
    exp_ap = load_exp("artemis_pbcp", seed)

    y_tn_all = clin_tn[target]
    train_samples = [s for s in exp_tn.index
                     if s in y_tn_all.index and not pd.isna(y_tn_all.loc[s])]
    test_samples = list(exp_ap.index.intersection(clin_ap.index))

    enc_dir = (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo"
               / "RCB.score" / EXPERIMENT / enc_key)
    if feat_set != "Rep" and cfg.get("norep"):
        z_tr = z_te = None
    else:
        z_tr = _encode(enc_dir, train_samples, exp_tn, seed)
        z_te = _encode(enc_dir, test_samples, exp_ap, seed)

    X_tr = _feature_matrix(feat_set, clin_tn, z_tr, cfg.get("confounders") or [],
                           train_samples)
    X_te = _feature_matrix(feat_set, clin_ap, z_te, cfg.get("confounders") or [],
                           test_samples)
    y_tr = y_tn_all.loc[train_samples].to_numpy()
    y_te = clin_ap.loc[test_samples, target].to_numpy()

    # Drop samples with NaN covariates (PiCoSK does this implicitly via the
    # data loader's nan handling — see Manual.__getitem__).
    tr_ok = ~np.isnan(X_tr).any(axis=1) & ~np.isnan(y_tr)
    te_ok = ~np.isnan(X_te).any(axis=1) & ~np.isnan(y_te)
    X_tr, y_tr = X_tr[tr_ok], y_tr[tr_ok]
    X_te, y_te = X_te[te_ok], y_te[te_ok]

    scaler = StandardScaler().fit(X_tr)
    X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

    if target == "RCB.score":
        reg = ElasticNet(alpha=cfg["alpha"], l1_ratio=cfg["l1_ratio"],
                         max_iter=10000, random_state=seed).fit(X_tr_s, y_tr)
        pred = reg.predict(X_te_s)
        return {
            "pred": pred, "y": y_te, "coef": reg.coef_,
            "intercept": reg.intercept_, "X_te_s": X_te_s,
            "test_spearmanr": spearmanr(pred, y_te)[0],
            "test_pearsonr": pearsonr(pred, y_te)[0],
            "test_rmse": float(np.sqrt(np.mean((pred - y_te) ** 2))),
        }
    # Classification (pCR) — keep y as `resp.pCR` (1=responder) to match
    # PiCoSK exactly. Pos_label=1 then naturally aligns with the paper's
    # rep_renamer / FI plotting convention.
    y_tr_b = y_tr.astype(int)
    y_te_b = y_te.astype(int)
    reg = LogisticRegression(
        penalty="elasticnet", C=1 / cfg["alpha"], l1_ratio=cfg["l1_ratio"],
        solver="saga", class_weight="balanced", max_iter=10000,
        random_state=seed,
    ).fit(X_tr_s, y_tr_b)
    pred_proba = reg.predict_proba(X_te_s)[:, 1]
    return {
        "pred": pred_proba, "y": y_te_b, "coef": reg.coef_[0],
        "intercept": reg.intercept_, "X_te_s": X_te_s,
        "test_auroc": roc_auc_score(y_te_b, pred_proba),
    }


test_rows, fit_cache_tn = [], {}
for target in ("RCB.score", "resp.pCR"):
    for feat_set, suffix, _ in FEATURE_SETS:
        for enc_key, rep_label in ENCODERS:
            for seed in SEEDS:
                try:
                    r = _fit_and_score(target, feat_set, suffix, None, enc_key, seed)
                except FileNotFoundError:
                    continue
                row = {"rep_type": rep_label, "feat_sets": feat_set,
                       "seed": seed, "target": target,
                       "model_type": "ElasticNet" if target == "RCB.score"
                                     else "LogisticRegression"}
                for k, v in r.items():
                    if k.startswith("test_"):
                        row[k] = v
                test_rows.append(row)
                fit_cache_tn[(target, feat_set, enc_key, seed)] = r

test_df = pd.DataFrame(test_rows).assign(dataset="test")
print("External-validation AUROC (pCR):")
print(test_df[test_df["target"] == "resp.pCR"]
      .groupby(["rep_type", "feat_sets"])["test_auroc"]
      .agg(["mean", "std"]).round(3))
print("\nExternal-validation Spearman ρ (RCB):")
print(test_df[test_df["target"] == "RCB.score"]
      .groupby(["rep_type", "feat_sets"])["test_spearmanr"]
      .agg(["mean", "std"]).round(3))

### 5-fold cross-validation on TransNEO

Same feature pipeline, but trained on 4/5 and held-out fold scored — summed
over 5 folds with `sklearn.model_selection.KFold(shuffle=True, random_state=4563)`,
matching the paper's single `hopt_seed = 4563`. Only the best-trial
hyperparameters are reused.

In [ ]:
from sklearn.model_selection import KFold


def _cv_metrics(target, feat_set, suffix, enc_key, hopt_seed=4563):
    reg_name = "ElasticNet" if target == "RCB.score" else "LogisticRegression"
    stage2_dir = (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo"
                  / target / EXPERIMENT / "pico" / f"{reg_name}_{enc_key}{suffix}")
    cfg = json.loads((stage2_dir / f"args_best_s{hopt_seed}.txt").read_text())

    exp_tn_h = load_exp("transneo", hopt_seed)
    y_tn_all = clin_tn[target]
    train_samples = np.array([s for s in exp_tn_h.index
                              if s in y_tn_all.index
                              and not pd.isna(y_tn_all.loc[s])])
    enc_dir = (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo"
               / "RCB.score" / EXPERIMENT / enc_key)
    if feat_set != "Rep" and cfg.get("norep"):
        z_full = None
    else:
        # Use the hopt_seed encoder + matching variance-sorted matrix.
        z_full = _encode(enc_dir, train_samples.tolist(), exp_tn_h, hopt_seed)

    X_full = _feature_matrix(feat_set, clin_tn, z_full,
                             cfg.get("confounders") or [], train_samples)
    y_full = y_tn_all.loc[train_samples].to_numpy()
    ok = ~np.isnan(X_full).any(axis=1) & ~np.isnan(y_full)
    X_full, y_full, samples = X_full[ok], y_full[ok], train_samples[ok]

    preds_all, y_all = [], []
    kf = KFold(n_splits=5, shuffle=True, random_state=hopt_seed)
    for tr_idx, va_idx in kf.split(X_full):
        scaler = StandardScaler().fit(X_full[tr_idx])
        X_tr_s, X_va_s = scaler.transform(X_full[tr_idx]), scaler.transform(X_full[va_idx])
        if target == "RCB.score":
            reg = ElasticNet(alpha=cfg["alpha"], l1_ratio=cfg["l1_ratio"],
                             max_iter=10000, random_state=hopt_seed
                            ).fit(X_tr_s, y_full[tr_idx])
            preds_all.append(reg.predict(X_va_s))
            y_all.append(y_full[va_idx])
        else:
            y_tr_b = y_full[tr_idx].astype(int)  # 1 = pCR (responder)
            reg = LogisticRegression(
                penalty="elasticnet", C=1 / cfg["alpha"],
                l1_ratio=cfg["l1_ratio"], solver="saga",
                class_weight="balanced", max_iter=10000,
                random_state=hopt_seed,
            ).fit(X_tr_s, y_tr_b)
            preds_all.append(reg.predict_proba(X_va_s)[:, 1])
            y_all.append(y_full[va_idx].astype(int))
    pred = np.concatenate(preds_all)
    y = np.concatenate(y_all)
    if target == "RCB.score":
        return {"val_spearmanr": spearmanr(pred, y)[0],
                "val_pearsonr": pearsonr(pred, y)[0],
                "val_rmse": float(np.sqrt(np.mean((pred - y) ** 2)))}
    return {"val_auroc": roc_auc_score(y, pred)}


val_rows = []
for target in ("RCB.score", "resp.pCR"):
    for feat_set, suffix, _ in FEATURE_SETS:
        for enc_key, rep_label in ENCODERS:
            try:
                m = _cv_metrics(target, feat_set, suffix, enc_key)
            except FileNotFoundError:
                continue
            m.update(rep_type=rep_label, feat_sets=feat_set, target=target,
                     model_type="ElasticNet" if target == "RCB.score"
                                else "LogisticRegression")
            val_rows.append(m)

val_df = pd.DataFrame(val_rows).assign(dataset="cv")

## 4. Figure 5c — RCB regression performance (paper plot code)

Two-panel pointplot — TransNEO 5-fold cross-validation on the left,
ARTemis+PBCP external validation on the right. PiCo (blue) above VAE
(orange); grey markers flag baselines that omit `z` (Clinical, RNA,
Clinical+RNA).

In [ ]:
plot_transneo_perf(val_df[val_df["target"] == "RCB.score"],
                   test_df[test_df["target"] == "RCB.score"],
                   target="RCB.score", model="ElasticNet", metric="spearmanr",
                   save_to=OUT_DIR / "fig5c_rcb_spearman")
plt.show()

## 5. Figure 5d — pCR classification performance

In [ ]:
plot_transneo_perf(val_df[val_df["target"] == "resp.pCR"],
                   test_df[test_df["target"] == "resp.pCR"],
                   target="resp.pCR", model="LogisticRegression", metric="auroc",
                   save_to=OUT_DIR / "fig5d_pcr_auroc")
plt.show()

## 6. Figure 5e/f — permutation feature importance (Clinical+z+RNA)

Same paper-style bar plot (`plot_feat_imps`) restricted to the
Clinical+`z`+RNA variant — the only configuration where `z`, clinical, and
RNA features can be compared head-to-head. Permutation is done live from
the fitted linear-model coefficients.

In [ ]:
import os
os.makedirs("figures", exist_ok=True)

NAMES_MAP = {
    "Danaher.Mast.cells": "Mast cell score",
    "PGR.log2.tpm": r"$\it{PGR}$ expression",
    "ESR1.log2.tpm": r"$\it{ESR1}$ expression",
    "ERBB2.log2.tpm": r"$\it{ERBB2}$ expression",
    "HER2.status": "HER2 status", "ER.status": "ER status",
    "Age.at.diagnosis": "Age at diagnosis",
    "LN.at.diagnosis": "LN involvement",
    "Grade.pre.chemotherapy": "Histological grade",
    "Histology": "Histological subtype",
    "TIDE.Exclusion": "T cell exclusion",
    "TIDE.Dysfunction": "T cell dysfunction",
    "Size.at.diagnosis": "Tumour size",
    "GGI.ssgsea.notnorm": "GGI score",
    "ESC.ssgsea.notnorm": "ES cell score",
    "Swanton.PaclitaxelScore": "Taxane score",
    "CytScore.log2": "Cytolytic score",
    "STAT1.ssgsea.notnorm": "STAT1 score",
}


def _fi_for(target, reg_name, metric, save_stub, classification):
    feat_set = "Clinical+Rep+RNA"
    cfg = json.loads(
        (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo" / target
         / EXPERIMENT / "pico" / f"{reg_name}_icovae_MCL1_16_Size.at.diagnosis_18"
         / f"args_best_s{SEEDS[0]}.txt").read_text()
    )
    confounders = cfg["confounders"]
    constraints = cfg["constraints"]
    # Pull the encoder's latent dim from the stage-1 args (z_dim = 32 for
    # the TransNEO MCL1_16 encoder; 16 constrained dims + 16 unconstrained).
    stage1_cfg = json.loads(
        (DEMO_ROOT / "data" / "outputs" / "depmap_gdsc_transneo" / "RCB.score"
         / EXPERIMENT / "icovae_MCL1_16" / "args_best.txt").read_text()
    )
    z_dim = stage1_cfg["z_dim"]
    # Mirror PiCoSK / rep_renamer column ordering: first len(constraints)
    # latent dims are renamed to z_<constraint_gene>, the rest stay as
    # bare integer indices (so plot_feat_imps' x.split('_')[1] yields '16',
    # '17', ...), then the 18 confounders as c_<col>.
    feature_names = (
        [f"z_{c}" for c in constraints]
        + [f"z_{i}" for i in range(len(constraints), z_dim)]
        + [f"c_{c}" for c in confounders]
    )

    fi_frames = []
    for seed in SEEDS:
        fit = fit_cache_tn[(target, feat_set, "icovae_MCL1_16", seed)]
        fi_frames.append(live_permutation_fi(
            fit["X_te_s"], fit["y"], feature_names=feature_names,
            coeffs=fit["coef"], intercept=fit["intercept"],
            seed=seed, n_iter=100, classification=classification,
        ))
    fi_df = pd.concat(fi_frames, axis=0, ignore_index=True)

    plot_feat_imps(
        fi_df, target=target, constraints=constraints,
        confounders=confounders, zdim=z_dim,
        enc="icovae_MCL1_16", reg=reg_name, experiment=EXPERIMENT,
        names_map=NAMES_MAP, metric=metric, save_path=save_stub,
        norep=False, sort_feats=True, top_k=16,
    )


_fi_for("RCB.score", "ElasticNet", "s",
        save_stub="transneo_rcb_clin_z_rna_s", classification=False)
plt.show()
_fi_for("resp.pCR", "LogisticRegression", "auroc",
        save_stub="transneo_pcr_clin_z_rna_auroc", classification=True)
plt.show()

# Mirror ./figures/fi_*.{png,svg} into demo/outputs/ for tidy collation.
import shutil
for stub in ("transneo_rcb_clin_z_rna_s", "transneo_pcr_clin_z_rna_auroc"):
    for ext in ("png", "svg"):
        src = Path("figures") / f"fi_{stub}.{ext}"
        if src.exists():
            shutil.copy(src, OUT_DIR / f"fig5_fi_{stub}.{ext}")

## 7. Save aggregated tables

In [ ]:
test_df.to_csv(OUT_DIR / "transneo_test_df.csv", index=False)
val_df.to_csv(OUT_DIR / "transneo_val_df.csv", index=False)